# Chapter 15: Decision Trees & Ensemble Learning
## Notebook Pendamping Praktik Interaktif

Notebook ini memandu Anda mengeksplorasi, memodelkan, memangkas (*pruning*), dan mengagregasi (*ensemble*) model pohon keputusan:
1. **Eksplorasi Dataset Telemetri Sensor Kerusakan Peralatan**.
2. **Baseline Decision Tree Classifier (Gini Impurity vs Entropy)**.
3. **Visualisasi Aturan Pohon Keputusan (IF-THEN Rules)**.
4. **Analisis Kriteria Pemangkasan Pohon (*Cost-Complexity Pruning*)**.
5. **Random Forest Classifier & Feature Importance (MDI)**.
6. **Evaluasi Out-of-Bag (OOB) vs 5-Fold Cross Validation**.
7. **Boosting: AdaBoost vs Gradient Boosting Classifier**.
8. **Evaluasi dan Analisis Matriks Konfusi Model Produksi**.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Muat dataset sensor industri
data_path = Path("../code/data/equipment_fault_sensor_data.csv")
df = pd.read_csv(data_path)
print(f"Dimensi Dataset: {df.shape[0]} baris x {df.shape[1]} kolom")
print(f"Distribusi Target: {dict(df['fault_category'].value_counts())}")
df.head(3)


Dimensi Dataset: 360 baris x 8 kolom
Distribusi Target: {'FAULT': np.int64(149), 'WARNING': np.int64(120), 'NORMAL': np.int64(91)}


## 2. Pemisahan Dataset Bebas Kebocoran (*Leak-Free Split*)
Dataset dibagi menjadi 75% Data Latih (270 sampel) dan 25% Data Uji (90 sampel) dengan stratifikasi kelas target.

In [2]:
features = ['temperature_c', 'vibration_mm_s', 'voltage_v', 'current_a', 'operating_hours', 'load_percentage']
X = df[features]
y = df['fault_category']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print(f"Data Latih: {X_train.shape[0]} sampel | Data Uji: {X_test.shape[0]} sampel")


Data Latih: 270 sampel | Data Uji: 90 sampel


## 3. Decision Tree: Gini Impurity vs Entropy
Mengevaluasi performa pohon keputusan tunggal dengan batasan kedalaman (`max_depth=4`).

In [3]:
tree_gini = DecisionTreeClassifier(criterion='gini', max_depth=4, random_state=42).fit(X_train, y_train)
acc_gini = accuracy_score(y_test, tree_gini.predict(X_test))

tree_ent = DecisionTreeClassifier(criterion='entropy', max_depth=4, random_state=42).fit(X_train, y_train)
acc_ent = accuracy_score(y_test, tree_ent.predict(X_test))

print(f"Akurasi Uji Gini Impurity (max_depth=4) : {acc_gini * 100:.2f}%")
print(f"Akurasi Uji Entropy/Info Gain (max_depth=4): {acc_ent * 100:.2f}%")


Akurasi Uji Gini Impurity (max_depth=4) : 92.22%
Akurasi Uji Entropy/Info Gain (max_depth=4): 90.00%


## 4. Visualisasi Aturan IF-THEN Pohon Keputusan
Menampilkan cabang keputusan logis yang dipelajari secara otomatis dari data sensor.

In [4]:
rules = export_text(tree_gini, feature_names=features, show_weights=True)
print(rules[:600])


|--- vibration_mm_s <= 4.73
|   |--- vibration_mm_s <= 3.53
|   |   |--- temperature_c <= 73.26
|   |   |   |--- voltage_v <= 369.71
|   |   |   |   |--- weights: [3.00, 0.00, 2.00] class: FAULT
|   |   |   |--- voltage_v >  369.71
|   |   |   |   |--- weights: [0.00, 67.00, 1.00] class: NORMAL
|   |   |--- temperature_c >  73.26
|   |   |   |--- current_a <= 35.76
|   |   |   |   |--- weights: [11.00, 0.00, 1.00] class: FAULT
|   |   |   |--- current_a >  35.76
|   |   |   |   |--- weights: [3.00, 0.00, 8.00] class: WARNING
|   |--- vibration_mm_s >  3.53
|   |   |--- temperature_c <= 84.01
|


## 5. Random Forest & Feature Importance (MDI)
Agregasi 100 pohon keputusan dengan *random feature subsampling* ($\sqrt{p}$) untuk mereduksi varians.

In [5]:
rf = RandomForestClassifier(n_estimators=100, max_features='sqrt', random_state=42).fit(X_train, y_train)
acc_rf = accuracy_score(y_test, rf.predict(X_test))
print(f"Akurasi Uji Random Forest (100 Trees): {acc_rf * 100:.2f}%")

feat_imp = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
print("\nFeature Importances (Mean Decrease in Impurity):\n", feat_imp)


Akurasi Uji Random Forest (100 Trees): 90.00%

Feature Importances (Mean Decrease in Impurity):
vibration_mm_s     0.383944
temperature_c      0.286829
voltage_v          0.088964
load_percentage    0.088415
operating_hours    0.083620
current_a          0.068227


## 6. Evaluasi Out-of-Bag (OOB) vs 5-Fold Stratified Cross-Validation

In [6]:
rf_oob = RandomForestClassifier(n_estimators=100, max_features='sqrt', oob_score=True, random_state=42).fit(X_train, y_train)
cv_scores = cross_val_score(rf_oob, X_train, y_train, cv=5)

print(f"Skor Out-of-Bag (OOB)       : {rf_oob.oob_score_ * 100:.2f}%")
print(f"5-Fold CV Score (Rata-rata) : {cv_scores.mean() * 100:.2f}% (+/- {cv_scores.std()*100:.2f}%)")
print(f"Hold-out Test Score         : {accuracy_score(y_test, rf_oob.predict(X_test)) * 100:.2f}%")


Skor Out-of-Bag (OOB)       : 95.56%
5-Fold CV Score (Rata-rata) : 94.07% (+/- 2.46%)
Hold-out Test Score         : 90.00%


## 7. Model Evaluasi Produksi: Gradient Boosting Classifier
Mengevaluasi model berkinerja tertinggi dengan laporan klasifikasi dan matriks konfusi.

In [7]:
gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42).fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)
print(f"Akurasi Uji Gradient Boosting: {accuracy_score(y_test, y_pred_gb) * 100:.2f}%")
print("\nLaporan Klasifikasi Detail:")
print(classification_report(y_test, y_pred_gb, digits=4))
print("Matriks Konfusi (NORMAL, WARNING, FAULT):")
print(confusion_matrix(y_test, y_pred_gb, labels=['NORMAL', 'WARNING', 'FAULT']))


Akurasi Uji Gradient Boosting: 93.33%

Laporan Klasifikasi Detail:
              precision    recall  f1-score   support

       FAULT     0.9024    1.0000    0.9487        37
      NORMAL     1.0000    0.9130    0.9545        23
     WARNING     0.9286    0.8667    0.8966        30

    accuracy                         0.9333        90
   macro avg     0.9437    0.9266    0.9333        90
weighted avg     0.9361    0.9333    0.9328        90

Matriks Konfusi (NORMAL, WARNING, FAULT):
[[21  2  0]
 [ 0 26  4]
 [ 0  0 37]]
